In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Example notebook demonstrating NdimSpline_JAX usage.

Shows how to build a jittable and auto-differentiable multidimensional
spline interpolant from gridded data.

@author: moteki
"""

import numpy as np
import jax.numpy as jnp
from jax import jit, grad, value_and_grad

from ndim_spline_jax import compute_coefs, make_interpolant

1. Define the grid information. For x-coordinates, we define the N-list of lower bounds a, the N-list b of upper bounds, and the N-list n of number of grid intervals.

In [2]:
#### synthetic data for demostration (5-dimension) ####
a=[0,0,0,0,0] # the user-defined lower bound of each x-coordinate [1st dim, ..., Nth dim]
b=[1,2,3,4,5]  # the user-defined upper bound of each x-coordinate [1st dim, ..., Nth dim]
n=[15,15,15,15,15] # the user-defined number of grid intervals in each x-coordinate [1st dim, ..., Nth dim]

2. Prepare an observation data y_data on the x gridpoints.

In [3]:
N = len(a)  # dimension N

# Generate gridded data using meshgrid (works for any N)
grids = [np.linspace(a[j], b[j], n[j] + 1) for j in range(N)]
mesh = np.meshgrid(*grids, indexing="ij")
y_data = np.ones_like(mesh[0])
for j in range(N):
    y_data *= np.sin(mesh[j])

3. Compute the spline coefficients from data using `compute_coefs`.

In [4]:
c = compute_coefs(N, jnp.array(y_data))

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


4. Create the JIT & AD -able interpolant using `make_interpolant`.

In [5]:
s = make_interpolant(a, b, n, c)

5. Use the generated interpolant with the `jax`'s JIT & Autograd functionalities.

In [6]:
# Specify a x-coordinate for function evaluation as a jnp array.
x = jnp.array([0.7, 1.0, 1.5, 2.0, 2.5])  # must satisfy a <= x <= b elementwise.

# Evaluate the interpolant
print(s(x))

# Gradient
ds = grad(s)
print(ds(x))

# Value and gradient together
s_fun = value_and_grad(s)
print(s_fun(x))

# JIT-compiled versions
s_jitted = jit(s)
print(s_jitted(x))

ds_jitted = jit(grad(s))
print(ds_jitted(x))

s_fun_jitted = jit(value_and_grad(s))
print(s_fun_jitted(x))

0.2942449462725687
[ 0.34933344  0.188933    0.02086654 -0.13466714 -0.39391559]
(Array(0.29424495, dtype=float64), Array([ 0.34933344,  0.188933  ,  0.02086654, -0.13466714, -0.39391559],      dtype=float64))
0.2942449462725687
[ 0.34933344  0.188933    0.02086654 -0.13466714 -0.39391559]
(Array(0.29424495, dtype=float64), Array([ 0.34933344,  0.188933  ,  0.02086654, -0.13466714, -0.39391559],      dtype=float64))


6. Compare the computation time of spline interpolant between non-Jitted and Jitted versions.

In [7]:
import timeit

%timeit s(x)             # function evaluation
%timeit s_jitted(x)      # function evaluation (jitted)
%timeit ds(x)            # gradient evaluation
%timeit ds_jitted(x)     # gradient evaluation (jitted)
%timeit s_fun(x)         # function and gradient evaluation
%timeit s_fun_jitted(x)  # function and gradient evaluation (jitted)

479 ms ± 12.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
80.2 μs ± 2.26 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
1.35 s ± 52.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
92.7 μs ± 4.46 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
1.43 s ± 58.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
97.7 μs ± 10.2 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
